# ODSB-16784: CPI Balancer Stability Multiplier

Calculate stability multipliers for Stonekey (스톤에이지 키우기) Android launch campaigns.

**Formula**: `AVG(pred[1] / normalizer[1])` from `bid.MODEL.prediction_logs` in the `imp` table.

- **ROAS campaigns**: `[I2I_TF_JOINT, ACTION, REVENUE]` — stability multiplier uses ACTION (offset 1)
- **CPA campaigns**: `[I2I_TF_JOINT, ACTION]` — stability multiplier uses ACTION (offset 1)

**Ref**: [CPI Balancer Doc](https://docs.google.com/document/d/14Egzm6BXQ9X-CFgovlEq4nGijo4WdkDFK2-X4aKZS0M) | [ODSB-16784](https://mlc.atlassian.net/browse/ODSB-16784)

In [ ]:
#@title Environment Setup

from google.cloud import bigquery
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

client = bigquery.Client(project='moloco-ods')

def run_query(sql):
    return client.query(sql).result().to_dataframe()

In [ ]:
#@title Campaign Configuration

ROAS_CAMPAIGNS = {
    'HvtBaZCEslMI1rJf': 'stonekey_launch_JP_And_tROAS_260303',
    'UdK42Uv8F2ZdIK5r': 'stonekey_launch_TW_And_tROAS_260303',
    'cLMYkp0auemwutTK': 'stonekey_launch_US_And_tROAS_260303',
    'huSii8ixFBbQjDXt': 'stonekey_launch_WW3_And_tROAS_260304',
    'jVkXbQ2LIdIKoaQg': 'stonekey_launch_WW2_And_tROAS_260303',
    'nazpxG3J5MareHRz': 'stonekey_launch_KR_And_tROAS_260303',
    'unbUzFob1WCmTrVD': 'stonekey_launch_WW1_And_tROAS_260303',
}

CPA_CAMPAIGNS = {
    'I1UFqtzeIhRkfCrK': 'stonekey_launch_TW_And_AEO(buy_pet_lv3)_260303',
    'NMrtO3Y5EN9EAiYm': 'stonekey_launch_DE_And_login_1st_260304',
    'NpO7UWUJVrOW83bd': 'stonekey_launch_NZ_And_login_1st_260304',
    'VsHwYSeUhcTRbuDQ': 'stonekey_launch_TW_And_login_1st_260303',
    'jfPWAJwwT0sXpYGv': 'stonekey_launch_SG_And_login_1st_260304',
    'znFCczWhy5JTsQHx': 'stonekey_launch_FR_And_login_1st_260304',
}

ALL_CAMPAIGNS = {**ROAS_CAMPAIGNS, **CPA_CAMPAIGNS}
print(f'ROAS campaigns: {len(ROAS_CAMPAIGNS)}  |  CPA campaigns: {len(CPA_CAMPAIGNS)}  |  Total: {len(ALL_CAMPAIGNS)}')

---
## 1. Calculate Stability Multipliers

Query `focal-elf-631.prod_stream_view.imp` for yesterday's impressions.

Stability multiplier = `AVG(pred / normalizer)` for the ACTION model (offset 1).

In [ ]:
#@title 1a. ROAS Campaign Stability Multipliers

roas_ids = ", ".join(f"'{c}'" for c in ROAS_CAMPAIGNS.keys())

query_roas = f"""
SELECT
  api.campaign.id AS campaign_id,
  COUNT(*) AS impressions,
  AVG(
    SAFE_DIVIDE(
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].pred,
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].wrapper.normalizer
    )
  ) AS stability_multiplier,
  STDDEV(
    SAFE_DIVIDE(
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].pred,
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].wrapper.normalizer
    )
  ) AS stability_std
FROM `focal-elf-631.prod_stream_view.imp`
WHERE DATE(timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  AND api.campaign.id IN ({roas_ids})
  AND bid.MODEL.prediction_logs[SAFE_OFFSET(1)].type = 'ACTION'
GROUP BY 1
"""

print('Querying ROAS campaign stability multipliers (yesterday)...')
df_roas = run_query(query_roas)
df_roas['campaign_name'] = df_roas['campaign_id'].map(ROAS_CAMPAIGNS)
df_roas['goal'] = 'ROAS'
df_roas = df_roas.sort_values('stability_multiplier', ascending=False)

print(f'\nROAS Campaigns ({len(df_roas)} results):')
print(f"{'Campaign':<45s}  {'Imps':>10s}  {'Multiplier':>11s}  {'Std':>8s}")
print(f"{'-'*45}  {'-'*10}  {'-'*11}  {'-'*8}")
for _, r in df_roas.iterrows():
    flag = ' ⚠️' if r['stability_multiplier'] > 2.0 or r['stability_multiplier'] < 0.5 else ''
    print(f"{r['campaign_name'][:45]:<45s}  {r['impressions']:>10,}  {r['stability_multiplier']:>11.4f}  {r['stability_std']:>8.4f}{flag}")

In [ ]:
#@title 1b. CPA Campaign Stability Multipliers

cpa_ids = ", ".join(f"'{c}'" for c in CPA_CAMPAIGNS.keys())

query_cpa = f"""
SELECT
  api.campaign.id AS campaign_id,
  COUNT(*) AS impressions,
  AVG(
    SAFE_DIVIDE(
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].pred,
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].wrapper.normalizer
    )
  ) AS stability_multiplier,
  STDDEV(
    SAFE_DIVIDE(
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].pred,
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].wrapper.normalizer
    )
  ) AS stability_std
FROM `focal-elf-631.prod_stream_view.imp`
WHERE DATE(timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  AND api.campaign.id IN ({cpa_ids})
  AND bid.MODEL.prediction_logs[SAFE_OFFSET(1)].type = 'ACTION'
GROUP BY 1
"""

print('Querying CPA campaign stability multipliers (yesterday)...')
df_cpa = run_query(query_cpa)
df_cpa['campaign_name'] = df_cpa['campaign_id'].map(CPA_CAMPAIGNS)
df_cpa['goal'] = 'CPA'
df_cpa = df_cpa.sort_values('stability_multiplier', ascending=False)

print(f'\nCPA Campaigns ({len(df_cpa)} results):')
print(f"{'Campaign':<45s}  {'Imps':>10s}  {'Multiplier':>11s}  {'Std':>8s}")
print(f"{'-'*45}  {'-'*10}  {'-'*11}  {'-'*8}")
for _, r in df_cpa.iterrows():
    flag = ' ⚠️' if r['stability_multiplier'] > 2.0 or r['stability_multiplier'] < 0.5 else ''
    print(f"{r['campaign_name'][:45]:<45s}  {r['impressions']:>10,}  {r['stability_multiplier']:>11.4f}  {r['stability_std']:>8.4f}{flag}")

---
## 2. Combined Summary

In [ ]:
#@title 2. Combined summary + flags

df_all = pd.concat([df_roas, df_cpa], ignore_index=True)
df_all = df_all[['goal', 'campaign_name', 'campaign_id', 'impressions', 'stability_multiplier', 'stability_std']]
df_all = df_all.sort_values(['goal', 'stability_multiplier'], ascending=[True, False])

# Flag anomalous values
df_all['flag'] = df_all['stability_multiplier'].apply(
    lambda x: 'HIGH' if x > 2.0 else ('LOW' if x < 0.5 else 'OK')
)

print(f"{'='*90}")
print(f"  STABILITY MULTIPLIER SUMMARY — Stonekey Launch Campaigns")
print(f"  Date: yesterday (DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY))")
print(f"  Typical range: 0.5 – 2.0")
print(f"{'='*90}")
print()
print(f"  {'Goal':<6s}  {'Campaign':<45s}  {'Imps':>10s}  {'Multiplier':>11s}  {'Std':>8s}  {'Flag':>5s}")
print(f"  {'-'*6}  {'-'*45}  {'-'*10}  {'-'*11}  {'-'*8}  {'-'*5}")
for _, r in df_all.iterrows():
    print(f"  {r['goal']:<6s}  {r['campaign_name'][:45]:<45s}  {r['impressions']:>10,}  "
          f"{r['stability_multiplier']:>11.4f}  {r['stability_std']:>8.4f}  {r['flag']:>5s}")

flagged = df_all[df_all['flag'] != 'OK']
if len(flagged) > 0:
    print(f"\n  ⚠️ {len(flagged)} campaign(s) outside typical range:")
    for _, r in flagged.iterrows():
        print(f"    - {r['campaign_name']}: {r['stability_multiplier']:.4f} ({r['flag']})")
else:
    print(f"\n  ✓ All campaigns within typical range (0.5–2.0).")

---
## 3. Multi-Day Trend (Optional)

Check stability multiplier trend over recent days to see if values are converging.

In [ ]:
#@title 3. Multi-day stability multiplier trend

LOOKBACK_DAYS = 3 #@param {type:"integer"}

all_ids = ", ".join(f"'{c}'" for c in ALL_CAMPAIGNS.keys())

query_trend = f"""
SELECT
  DATE(timestamp) AS date,
  api.campaign.id AS campaign_id,
  COUNT(*) AS impressions,
  AVG(
    SAFE_DIVIDE(
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].pred,
      bid.MODEL.prediction_logs[SAFE_OFFSET(1)].wrapper.normalizer
    )
  ) AS stability_multiplier
FROM `focal-elf-631.prod_stream_view.imp`
WHERE DATE(timestamp) BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {LOOKBACK_DAYS} DAY)
                           AND DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  AND api.campaign.id IN ({all_ids})
  AND bid.MODEL.prediction_logs[SAFE_OFFSET(1)].type = 'ACTION'
GROUP BY 1, 2
ORDER BY 2, 1
"""

print(f'Querying {LOOKBACK_DAYS}-day trend...')
df_trend = run_query(query_trend)
df_trend['campaign_name'] = df_trend['campaign_id'].map(ALL_CAMPAIGNS)
df_trend['goal'] = df_trend['campaign_id'].apply(lambda x: 'ROAS' if x in ROAS_CAMPAIGNS else 'CPA')

# Pivot for display
pivot_trend = df_trend.pivot_table(
    index=['goal', 'campaign_name'], columns='date',
    values='stability_multiplier', aggfunc='first'
).round(4)

print(f'\nStability Multiplier Trend ({LOOKBACK_DAYS} days):')
display(pivot_trend)